# 01 · Images, patches and a tiny visual encoder

*Vision-Language Models (VLMs)*, notebook 1 of 4

A vision-language model learns a relationship between pictures and words. Two things must be right before it can: the data must pair each image with language that really describes it, and the image must be turned into vectors that a network can compute with. In this notebook you generate a small paired dataset whose correct answers you know exactly, and you build the first half of a vision-language model, the visual encoder, one typed function at a time.

## What you will build

- a generator of 16 × 16 pictures of coloured bars, each paired with a caption such as "a large red horizontal bar", with four attribute combinations held back for testing;
- `normalise_images`: per-channel normalisation with statistics measured on the training images only;
- `patchify` and `unpatchify`: cut images into 4 × 4 patches, and put the patches back together;
- `embed_patches`: a linear patch embedding with a learned position vector for every patch;
- `encode_images`: a tiny encoder that turns 16 patches into patch features and one pooled image vector.

## Learning objectives

By the end of this notebook you will be able to:

1. design paired image–caption data with a split that tests new combinations of familiar attributes;
2. explain why normalisation statistics come from the training images only, and implement the normalisation;
3. convert between images of shape (n, H, W, C) and patch sequences of shape (n, N, P·P·C), and count patches for any resolution;
4. implement a patch embedding with positions and a pooled encoder with the tiny tensor engine, with gradients that a check confirms;
5. measure what the pooled image vector keeps about colour, size and orientation, on combinations seen in training and on held-out combinations.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the pictures. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: every image is drawn by the generator in this notebook.
- Training uses the tiny tensor engine you built in Base Model Training. A hidden recap cell defines it.
- Every random number comes from a seeded generator, so your numbers match the ones quoted in the text up to tiny floating-point differences. The whole notebook runs in a few seconds.

## Setup

Run this cell first. It imports NumPy, `time` and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below defines the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (a NumPy array that remembers how it was computed, so that `backward()` can fill in `.grad`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it is gradient-checked. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

### Checking functions that build Tensors

`practice` compares plain values. Several functions in this notebook take `Tensor`s and return `Tensor`s, and for those it is not enough that the numbers come out right: the gradients must also flow back to the weights, or training silently stops working. `practice_tensor` runs your function and the reference on fresh copies of the same inputs, compares the values, then calls `backward()` and compares the gradient of every input that has `requires_grad=True`. Read it once; you will use it like `practice`.

In [ ]:
def _fresh(value):
    """A copy of every Tensor inside value, so each check starts from clean gradients."""
    if isinstance(value, Tensor):
        return Tensor(value.data.copy(), requires_grad=value.requires_grad)
    if isinstance(value, dict):
        return {key: _fresh(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(_fresh(item) for item in value)
    return value


def _trainable_leaves(value):
    if isinstance(value, Tensor):
        return [value] if value.requires_grad else []
    if isinstance(value, dict):
        return [leaf for key in sorted(value) for leaf in _trainable_leaves(value[key])]
    if isinstance(value, (list, tuple)):
        return [leaf for item in value for leaf in _trainable_leaves(item)]
    return []


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """practice() for functions that build Tensors.

    Each case runs on fresh copies of its Tensor arguments. The check compares
    the values your function returns AND the gradients that flow back to every
    argument with requires_grad=True, so a function that cuts the graph (for
    example by working on .data) is caught as well.
    """
    def checked(function):
        def run(*args):
            copies = [_fresh(arg) for arg in args]
            out = function(*copies)
            outs = out if isinstance(out, tuple) else (out,)
            if not all(isinstance(item, Tensor) for item in outs):
                raise TypeError("return Tensor results built from Tensor operations, so gradients can flow")
            total = Tensor(0.0)
            for index, item in enumerate(outs):
                weights = np.random.default_rng(index).normal(size=item.shape)
                total = total + (item * weights).sum()
            total.backward()
            grads = [leaf.grad if leaf.grad is not None else np.zeros_like(leaf.data)
                     for leaf in _trainable_leaves(copies)]
            return {"values": [item.data for item in outs], "gradients": grads}
        return run

    mine, ref = checked(yours), checked(reference)
    return yours if practice(name, mine, ref, cases, atol) is mine else reference

## Part 1 · Paired data you can trust

### Images whose captions are true by construction

Web-scale vision-language data pairs millions of photographs with text found next to them, and much of that text describes something other than what the picture shows. To see every mechanism clearly, this course uses images whose captions are true by construction. Each image is 16 × 16 pixels with three colour channels and shows one bar on a dark, slightly noisy background. Three attributes describe the bar:

| attribute | values |
|---|---|
| colour | red, green, blue, yellow |
| size | small (2 × 8 pixels), large (3 × 12 pixels) |
| orientation | horizontal, vertical |

That makes $4 \times 2 \times 2 = 16$ combinations and 16 possible captions. The generator first writes a **scene**: a small record of attribute indices, the bar's top-left corner and a noise seed. Only then does it draw the pixels. Keeping the scene beside the image is this notebook's version of a data manifest: you can always check that a caption describes the pixels, and in notebook 4 you will redraw a scene with exactly one attribute changed.

**Shapes.** `render` returns images of shape (n, 16, 16, 3): height and width first, then the three channels, with values between 0 and 1. NumPy and Matplotlib use this channels-last layout; frameworks such as PyTorch usually store a batch as (n, 3, 16, 16) instead.

The next cell is given: read it, but you do not need to type it.

In [ ]:
COLOURS = ("red", "green", "blue", "yellow")
SIZES = ("small", "large")
ORIENTATIONS = ("horizontal", "vertical")
ATTRIBUTES = {"colour": COLOURS, "size": SIZES, "orientation": ORIENTATIONS}
RGB = np.array([[0.85, 0.12, 0.10],   # red
                [0.15, 0.70, 0.20],   # green
                [0.20, 0.30, 0.90],   # blue
                [0.90, 0.80, 0.10]])  # yellow
BAR = {0: (2, 8), 1: (3, 12)}  # size index -> (thickness, length) in pixels
IMAGE_SIZE = 16

# Every (colour, size, orientation) combination, as attribute indices.
ALL_COMBOS = [(c, s, o) for c in range(4) for s in range(2) for o in range(2)]
# Four combinations never shown in training. Every single attribute value
# still appears in training; only these particular combinations are new.
HELD_OUT_COMBOS = [(0, 0, 1), (1, 1, 0), (2, 0, 0), (3, 1, 1)]
TRAIN_COMBOS = [combo for combo in ALL_COMBOS if combo not in HELD_OUT_COMBOS]


def bar_box(size, orientation):
    """(height, width) of the bar in pixels."""
    thickness, length = BAR[int(size)]
    return (thickness, length) if orientation == 0 else (length, thickness)


def sample_scenes(n, combos, seed):
    """Describe n images: attribute indices, the bar's top-left corner and a noise seed."""
    rng = np.random.default_rng(seed)
    chosen = np.array(combos)[rng.integers(0, len(combos), n)]
    scenes = {"colour": chosen[:, 0], "size": chosen[:, 1], "orientation": chosen[:, 2]}
    boxes = np.array([bar_box(s, o) for s, o in zip(scenes["size"], scenes["orientation"])])
    scenes["row"] = rng.integers(0, IMAGE_SIZE - boxes[:, 0] + 1)
    scenes["col"] = rng.integers(0, IMAGE_SIZE - boxes[:, 1] + 1)
    scenes["noise_seed"] = rng.integers(0, 2**31, n)
    return scenes


def render(scenes):
    """Draw every scene as a (16, 16, 3) RGB image with values in [0, 1]."""
    n = len(scenes["colour"])
    images = np.empty((n, IMAGE_SIZE, IMAGE_SIZE, 3))
    for i in range(n):
        rng = np.random.default_rng(int(scenes["noise_seed"][i]))
        image = 0.08 + 0.06 * rng.random((IMAGE_SIZE, IMAGE_SIZE, 3))  # dark, slightly noisy background
        shade = rng.uniform(0.85, 1.15)  # brightness varies from image to image
        tint = rng.uniform(-0.1, 0.1, 3)  # and so does the exact hue
        colour = RGB[scenes["colour"][i]] * shade + tint
        height, width = bar_box(scenes["size"][i], scenes["orientation"][i])
        r, c = scenes["row"][i], scenes["col"][i]
        image[r:r + height, c:c + width] = colour + 0.04 * rng.normal(size=(height, width, 3))
        images[i] = np.clip(image, 0.0, 1.0)
    return images


def describe(scenes):
    """The caption of every scene, for example 'a large red horizontal bar'."""
    return [f"a {SIZES[s]} {COLOURS[c]} {ORIENTATIONS[o]} bar"
            for c, s, o in zip(scenes["colour"], scenes["size"], scenes["orientation"])]


def combo_ids(scenes):
    """One number from 0 to 15 per scene: which of the 16 combinations it shows."""
    return scenes["colour"] * 4 + scenes["size"] * 2 + scenes["orientation"]

In [ ]:
demo_scenes = sample_scenes(8, ALL_COMBOS, seed=7)
demo_images = render(demo_scenes)
fig, axes = plt.subplots(1, 8, figsize=(12, 2.2))
for ax, image, caption in zip(axes, demo_images, describe(demo_scenes)):
    ax.imshow(image, interpolation="nearest")
    ax.set_title("\n".join(caption.split()[1:4]), fontsize=8)
    ax.axis("off")
plt.show()
print("images:", demo_images.shape, "values from", demo_images.min().round(3), "to", demo_images.max().round(3))
print("first scene:", {key: int(value[0]) for key, value in demo_scenes.items()})

**Expected behaviour.** Eight small pictures appear, each titled with the size, colour and orientation from its caption, and the images have shape (8, 16, 16, 3) with values between 0 and 1. The shade and hue of each colour vary slightly from image to image, so that a model cannot succeed by memorising one exact pixel value. Compare a few titles with the pictures: this is the audit a real dataset needs, and it is cheap here only because the captions are generated from the scenes.

### Hold out combinations, not just images

A random split of images tests whether a model recognises new pictures of familiar combinations. A vision-language model should also cope with combinations it has never seen as a whole, such as a small red vertical bar when training only showed red bars that were large or horizontal. The CoGenT split of the CLEVR benchmark tests this kind of generalisation in the same way. Here the four combinations in `HELD_OUT_COMBOS` never appear in training, although every colour, both sizes and both orientations do.

The next cell draws three sets with separate seeds and keeps each image's scene as its manifest row:

- **train**: 800 images of the 12 training combinations;
- **seen**: 300 new images of the same 12 combinations;
- **held-out**: 300 images of the 4 held-out combinations.

In a real project the unit you split by is usually the source: a photograph, a document or a recording session. Near-duplicates from one source must stay in the same set, or the test becomes easier than real use.

In [ ]:
train_scenes = sample_scenes(800, TRAIN_COMBOS, seed=101)
seen_scenes = sample_scenes(300, TRAIN_COMBOS, seed=202)        # new images, combinations seen in training
held_out_scenes = sample_scenes(300, HELD_OUT_COMBOS, seed=303)  # new images, combinations never seen
train_images, seen_images, held_out_images = render(train_scenes), render(seen_scenes), render(held_out_scenes)


print("id  split      caption")
for name, scenes in (("train", train_scenes), ("seen", seen_scenes), ("held-out", held_out_scenes)):
    for i, caption in enumerate(describe(scenes)[:2]):
        print(f"{i:<3d} {name:10s} {caption}")

train_set = set(combo_ids(train_scenes).tolist())
held_out_set = set(combo_ids(held_out_scenes).tolist())
assert not train_set & held_out_set, "a held-out combination leaked into the training set"
assert len(train_set) == len(TRAIN_COMBOS) and len(held_out_set) == len(HELD_OUT_COMBOS)
print(f"training combinations: {len(train_set)}; held-out combinations: {sorted(held_out_set)} (never in training)")

**Expected behaviour.** The listing shows two manifest rows from each set, and the check confirms that the 12 training combinations and the 4 held-out ones do not overlap. The held-out ids are 1, 6, 8 and 15: for example id 1 is colour 0, size 0, orientation 1, a small red vertical bar.

### 🧪 Try it: a centre crop can delete the evidence

Image pipelines often crop. A crop that removes part of the bar can make a true caption false: a large bar cut in half looks small, and a bar cropped away entirely has no colour at all. The cell below draws each training bar's footprint as a mask of pixels and counts how many of them survive a centred square crop. Change `crop`, run the cell again, and watch how much evidence each setting removes.

In [ ]:
crop = 12  # try 16, 14 and 10, then run this cell again
start = (IMAGE_SIZE - crop) // 2
window = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
window[start:start + crop, start:start + crop] = True
lost = np.zeros(len(train_scenes["colour"]))
for i in range(len(lost)):
    height, width = bar_box(train_scenes["size"][i], train_scenes["orientation"][i])
    bar = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
    bar[train_scenes["row"][i]:train_scenes["row"][i] + height, train_scenes["col"][i]:train_scenes["col"][i] + width] = True
    lost[i] = 1 - (bar & window).sum() / bar.sum()
print(f"a centred {crop} x {crop} crop: {np.mean(lost > 0):.0%} of training bars lose pixels, "
      f"{np.mean(lost > 0.5):.0%} lose more than half")

**Expected behaviour.** With `crop = 16` nothing is lost. With `crop = 12` about three bars in four lose some pixels and nearly one in five loses more than half, so some "large" captions would now describe bars that look small. The fix is not to hope the model guesses: choose a crop and resolution that keep the evidence the task needs, or remove the examples whose labels the preprocessing has made false.

## Part 2 · Normalise with training statistics

Networks train more easily when their inputs are centred near zero with a similar spread in every channel. Per-channel normalisation subtracts each channel's mean and divides by its standard deviation:

$$x'_{n,h,w,c} = \frac{x_{n,h,w,c} - \mu_c}{\sigma_c}, \qquad c \in \{\text{red}, \text{green}, \text{blue}\}.$$

The statistics $\mu$ and $\sigma$ (each of shape (3,)) are measured over every pixel of the **training** images only, then reused unchanged for the seen and held-out images. Measuring them on test images would leak information about the test set into preprocessing. A pretrained encoder goes further: it expects the exact constants it was trained with, which its model card publishes, and arbitrary replacements can degrade it even though every shape still matches.

`channel_stats` is given; it averages over the image, row and column axes:

In [ ]:
def channel_stats(images):
    """Mean and standard deviation of each colour channel, over every pixel of every image."""
    return images.mean(axis=(0, 1, 2)), images.std(axis=(0, 1, 2))


mean, std = channel_stats(train_images)
print("channel means:", mean.round(3), " channel standard deviations:", std.round(3))

### ✍️ Your turn: `normalise_images`

Write `normalise_images(images, mean, std)`.

- `images`: an array of shape (n, H, W, 3);
- `mean`, `std`: arrays of shape (3,), one value per channel;
- return a new array of shape (n, H, W, 3) holding $(x - \mu_c)/\sigma_c$. Do not modify `images`.

*Hint:* NumPy broadcasting lines shapes up from the right, so (n, H, W, 3) and (3,) combine channel by channel without any loop.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def normalise_images_reference(images, mean, std):
    """(images - mean) / std for every channel. images: (n, H, W, 3); mean, std: (3,)."""
    return (images - mean) / std

In [ ]:
def normalise_images(images, mean, std):
    """(images - mean) / std for every channel. images: (n, H, W, 3); mean, std: (3,)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(11)
small_images = case_rng.random((2, 4, 4, 3))
normalise_images = practice("normalise_images", normalise_images, normalise_images_reference, cases=[
    (small_images, np.array([0.5, 0.4, 0.3]), np.array([0.2, 0.25, 0.5])),
    (train_images[:3], mean, std),
    (small_images[:1], np.zeros(3), np.ones(3)),  # edge case: one image, statistics that change nothing
])

**Expected behaviour.** ✅ on three cases, including a single image with mean 0 and standard deviation 1, which must come back unchanged. The next cell confirms the defining property on the full training set: after normalisation every channel has mean 0 and standard deviation 1. The held-out images are normalised with the same training statistics, so their channel means need not be zero: the held-out set contains a different mix of colours and bar sizes, and that difference is real information, not an error to remove.

In [ ]:
train_normalised = normalise_images(train_images, mean, std)
print("training channel means:", train_normalised.mean(axis=(0, 1, 2)).round(6))
print("training channel stds: ", train_normalised.std(axis=(0, 1, 2)).round(6))
print("held-out channel means:", normalise_images(held_out_images, mean, std).mean(axis=(0, 1, 2)).round(3))

## Part 3 · Cut images into patches

A patch-based encoder, such as the vision transformer (ViT), treats an image as a sequence of small square patches. With patch side $P$, an $H \times W$ image gives a grid of $H/P$ rows and $W/P$ columns of patches:

$$N = \frac{H}{P} \cdot \frac{W}{P}, \qquad \text{patch length} = P \cdot P \cdot C.$$

For our 16 × 16 × 3 images and $P = 4$ that is $N = 16$ patches of $4 \cdot 4 \cdot 3 = 48$ numbers each, so a batch of shape (n, 16, 16, 3) becomes (n, 16, 48). A 224 × 224 photograph with 16-pixel patches gives $14 \times 14 = 196$ patches.

**Order matters.** Number the patches row by row, like reading text: patch 0 is the top-left corner, patch 1 is to its right, and patch $H/P$ starts the second row. Inside a patch, list the values row by row, then column by column, then channel by channel. The demo shows the first two patches of a 4 × 4 single-channel image whose pixels are numbered 0 to 15, using plain slicing:

In [ ]:
tiny = np.arange(16.0).reshape(1, 4, 4, 1)  # one image, 4 x 4 pixels, one channel
print(tiny[0, :, :, 0])
print("patch 0:", tiny[0, 0:2, 0:2, 0].ravel(), "  patch 1:", tiny[0, 0:2, 2:4, 0].ravel())

### ✍️ Your turn: `patchify`

Write `patchify(images, patch_size)`.

- `images`: shape (n, H, W, C); `patch_size` $P$ divides both H and W;
- return shape (n, N, P·P·C) with $N = (H/P)(W/P)$, patches numbered row by row and the values inside each patch ordered row, column, channel, exactly as in the demo.

*Hints:* reshape to six axes, (n, H/P, P, W/P, P, C), so that each pixel index splits into "which patch" and "where inside it". Then `transpose(0, 1, 3, 2, 4, 5)` brings the two patch-grid axes together before a final reshape. A loop over patch rows and columns also works, just more slowly.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def patchify_reference(images, patch_size):
    """Cut images (n, H, W, C) into non-overlapping square patches: (n, N, P*P*C), N = (H/P)*(W/P).

    Patches are numbered row by row; inside a patch the values run row by row,
    then column by column, then channel by channel.
    """
    n, height, width, channels = images.shape
    p = patch_size
    grid = images.reshape(n, height // p, p, width // p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, cols, p, p, C)
    return grid.reshape(n, (height // p) * (width // p), p * p * channels)

In [ ]:
def patchify(images, patch_size):
    """Cut images (n, H, W, C) into non-overlapping square patches: (n, N, P*P*C), N = (H/P)*(W/P).

    Patches are numbered row by row; inside a patch the values run row by row,
    then column by column, then channel by channel.
    """
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(12)
patchify = practice("patchify", patchify, patchify_reference, cases=[
    (np.arange(16.0).reshape(1, 4, 4, 1), 2),
    (case_rng.random((2, 8, 8, 3)), 4),
    (case_rng.random((1, 4, 4, 3)), 4),  # edge case: a single patch covers the whole image
])
print(patchify(np.arange(16.0).reshape(1, 4, 4, 1), 2)[0])

**Expected behaviour.** ✅, and the printed array has one row per patch: `[0, 1, 4, 5]`, `[2, 3, 6, 7]`, `[8, 9, 12, 13]`, `[10, 11, 14, 15]`. A common bug is reshaping straight to (n, N, P·P·C) without the transpose: the shapes are right, but each "patch" is then a strip of one image row, and the first row would read `[0, 1, 2, 3]`.

### ✍️ Your turn: `unpatchify`

Write `unpatchify(patches, patch_size, height, width)`, the exact inverse of `patchify`.

- `patches`: shape (n, N, P·P·C) with $N = (H/P)(W/P)$;
- return the images, shape (n, H, W, C), where $C$ = patch length / (P·P).

*Hint:* undo `patchify`'s steps in reverse order. Reshape to (n, H/P, W/P, P, P, C); the same `transpose(0, 1, 3, 2, 4, 5)` swaps the middle axes back; then reshape to (n, H, W, C).

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def unpatchify_reference(patches, patch_size, height, width):
    """The inverse of patchify: (n, N, P*P*C) back to images (n, H, W, C)."""
    n, _, patch_dim = patches.shape
    p = patch_size
    channels = patch_dim // (p * p)
    grid = patches.reshape(n, height // p, width // p, p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, p, cols, p, C)
    return grid.reshape(n, height, width, channels)

In [ ]:
def unpatchify(patches, patch_size, height, width):
    """The inverse of patchify: (n, N, P*P*C) back to images (n, H, W, C)."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
unpatchify = practice("unpatchify", unpatchify, unpatchify_reference, cases=[
    (patchify(np.arange(16.0).reshape(1, 4, 4, 1), 2), 2, 4, 4),
    (patchify(case_rng.random((2, 8, 8, 3)), 4), 4, 8, 8),
    (case_rng.random((1, 1, 48)), 4, 4, 4),  # edge case: one patch becomes one 4 x 4 x 3 image
])

Now cut the real images. `prepare` (given) normalises with the training statistics and then patchifies, the full preprocessing path from now on. The round trip `unpatchify(patchify(x))` must return `x` exactly, because both functions only move numbers around. The plot shows one training image cut into its 16 patches.

In [ ]:
def prepare(images, mean, std, patch_size=4):
    """Normalise with training statistics, then cut into patches."""
    return patchify(normalise_images(images, mean, std), patch_size)


train_patches = prepare(train_images, mean, std)
print("patches:", train_patches.shape)
assert np.array_equal(unpatchify(patchify(train_images, 4), 4, 16, 16), train_images), "the round trip must be exact"

raw_patches = patchify(train_images[:1], 4)[0]  # unnormalised, so the colours display correctly
fig, axes = plt.subplots(4, 4, figsize=(3.2, 3.4))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(raw_patches[k].reshape(4, 4, 3), interpolation="nearest")
    ax.set_title(str(k), fontsize=7)
    ax.axis("off")
fig.suptitle(describe(train_scenes)[0], fontsize=9)
plt.show()

**Expected behaviour.** The patches have shape (800, 16, 48), the round trip is exact, and the grid shows the first training image split into patches 0 to 15, numbered row by row. Notice how few patches contain any of the bar: most of the image is background.

## Part 4 · Embed the patches, with positions

Each patch is a vector of 48 numbers. A **linear patch embedding** maps every patch to a vector of width $D$ with one weight matrix shared by all patches, and adds a learned **position vector** for the patch's place in the grid:

$$\mathbf{z}_{i} = \mathbf{x}_{i} W + \mathbf{b} + \mathbf{p}_{i}, \qquad W \in \mathbb{R}^{48 \times D},\ \mathbf{b} \in \mathbb{R}^{D},\ \mathbf{p}_i \in \mathbb{R}^{D}.$$

Without $\mathbf{p}_i$, a patch in the top-left corner and the same patch in the bottom-right corner would get identical vectors, and the encoder would see an unordered bag of patches. Shapes: patches (n, 16, 48) → embeddings (n, 16, D). This notebook uses $D = 32$. The given cell creates the encoder's weights and counts them:

In [ ]:
def init_image_encoder(seed, patch_dim=48, n_patches=16, width=32):
    """Weights of the tiny image encoder, all trainable Tensors."""
    rng = np.random.default_rng(seed)
    return {
        "W_patch": parameter(patch_dim, width, rng=rng),
        "b_patch": zeros_parameter(width),
        "pos": parameter(n_patches, width, rng=rng, scale=0.1),
        "W_mix": parameter(width, width, rng=rng),
        "b_mix": zeros_parameter(width),
    }


encoder = init_image_encoder(seed=0)
for name, value in encoder.items():
    print(f"{name:8s} shape {value.shape}")
print("parameters:", sum(value.data.size for value in encoder.values()))

### ✍️ Your turn: `embed_patches`

Write `embed_patches(patches, W, b, pos)`.

- `patches`: a `Tensor` of shape (n, N, patch_dim);
- `W`: (patch_dim, D); `b`: (D,); `pos`: (N, D), all `Tensor`s;
- return the `Tensor` $\mathbf{x}W + \mathbf{b} + \mathbf{p}$ of shape (n, N, D).

*Hints:* `@` multiplies the last axis of `patches` with `W` for every image and patch at once, and both `b` and `pos` broadcast over the leading axes. Use only `Tensor` operations, never `.data`, so that the gradients reach `W`, `b` and `pos`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def embed_patches_reference(patches, W, b, pos):
    """Linear patch embedding plus one learned position vector per patch.

    patches: Tensor (n, N, patch_dim); W: (patch_dim, D); b: (D,); pos: (N, D).
    Returns a Tensor (n, N, D).
    """
    return patches @ W + b + pos

In [ ]:
def embed_patches(patches, W, b, pos):
    """Linear patch embedding plus one learned position vector per patch.

    patches: Tensor (n, N, patch_dim); W: (patch_dim, D); b: (D,); pos: (N, D).
    Returns a Tensor (n, N, D).
    """
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(13)
embed_patches = practice_tensor("embed_patches", embed_patches, embed_patches_reference, cases=[
    (Tensor(case_rng.normal(size=(2, 4, 6))), parameter(6, 5, rng=case_rng), parameter(5, rng=case_rng, scale=0.1),
     parameter(4, 5, rng=case_rng, scale=0.1)),
    (Tensor(train_patches[:2]), encoder["W_patch"], encoder["b_patch"], encoder["pos"]),
    (Tensor(case_rng.normal(size=(1, 1, 6))), parameter(6, 5, rng=case_rng), parameter(5, rng=case_rng, scale=0.1),
     parameter(1, 5, rng=case_rng, scale=0.1)),  # edge case: one image with one patch
])

**Expected behaviour.** ✅ on three cases, including the real shapes (2, 16, 48) → (2, 16, 32). If you leave out `pos`, the values differ and the check shows ❌. If you compute with `.data`, the values match but the gradients come back as zeros, and the check shows ❌ for the gradients.

## Part 5 · A tiny encoder that pools the patches

The encoder applies one ReLU layer to every patch embedding separately and then averages over the patches:

$$\mathbf{h}_i = \operatorname{ReLU}(\mathbf{z}_i W_{\text{mix}} + \mathbf{b}_{\text{mix}}), \qquad \mathbf{g} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{h}_i.$$

It returns both: the **patch features** $\mathbf{h}$, shape (n, 16, 32), which keep one vector per place in the image, and the **pooled** vector $\mathbf{g}$, shape (n, 32), which summarises the whole image. Retrieval and classification usually use a pooled vector; grounding, reading small text and questions about where things are need the spatial features. Real vision transformers put self-attention layers between these steps, so that every patch vector can draw on every other patch. This encoder skips them to stay fast, and Experiment 1 shows one thing that costs.

`Tensor.mean(axis=1)` averages over the patch axis:

In [ ]:
three_patches = Tensor(np.arange(12.0).reshape(1, 3, 4))  # one image, 3 patches, width 4
print(three_patches.mean(axis=1).data)  # the average of the three rows

### ✍️ Your turn: `encode_images`

Write `encode_images(patches, params)`.

- `patches`: a `Tensor` of shape (n, N, patch_dim);
- `params`: the dictionary from `init_image_encoder`, with keys `W_patch`, `b_patch`, `pos`, `W_mix` and `b_mix`;
- return the tuple `(features, pooled)`: `features` is $\operatorname{ReLU}(\mathbf{z} W_{\text{mix}} + \mathbf{b}_{\text{mix}})$ of shape (n, N, D), where $\mathbf{z}$ is your `embed_patches` output, and `pooled` is the mean of `features` over the patch axis, shape (n, D).

*Hint:* three lines: call `embed_patches`, apply the layer with `.relu()`, then `.mean(axis=1)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def encode_images_reference(patches, params):
    """The tiny encoder: embed the patches, apply a ReLU layer to each patch, then average.

    patches: Tensor (n, N, patch_dim); params: the dictionary from init_image_encoder.
    Returns (features, pooled): Tensors of shape (n, N, D) and (n, D).
    """
    tokens = embed_patches(patches, params["W_patch"], params["b_patch"], params["pos"])
    features = (tokens @ params["W_mix"] + params["b_mix"]).relu()
    return features, features.mean(axis=1)

In [ ]:
def encode_images(patches, params):
    """The tiny encoder: embed the patches, apply a ReLU layer to each patch, then average.

    patches: Tensor (n, N, patch_dim); params: the dictionary from init_image_encoder.
    Returns (features, pooled): Tensors of shape (n, N, D) and (n, D).
    """
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
small_params = init_image_encoder(seed=3, patch_dim=6, n_patches=4, width=5)
encode_images = practice_tensor("encode_images", encode_images, encode_images_reference, cases=[
    (Tensor(case_rng.normal(size=(2, 4, 6))), small_params),
    (Tensor(train_patches[:3]), encoder),
    (Tensor(case_rng.normal(size=(1, 4, 6))), small_params),  # edge case: a batch of one image
])
features, pooled = encode_images(Tensor(train_patches[:5]), encoder)
print("features:", features.shape, " pooled:", pooled.shape)

**Expected behaviour.** ✅ on three cases, then `features: (5, 16, 32)` and `pooled: (5, 32)`. Because of the ReLU, many entries of `features` are exactly zero; the pooled vector averages them, so it is non-negative too.

## 🧪 Experiments

### 🧪 Experiment 1: what the position vectors add

Each patch passes through the encoder on its own, and the mean ignores order. So **without** position vectors, shuffling an image's patches cannot change its pooled vector: the encoder would give the same summary for the real image and for a scrambled jigsaw of it. With position vectors, the same patch content produces different features in different places. The cell checks both claims on four training images.

In [ ]:
shuffle = np.random.default_rng(5).permutation(16)
original, scrambled = Tensor(train_patches[:4]), Tensor(train_patches[:4][:, shuffle])
no_positions = dict(encoder, pos=Tensor(np.zeros((16, 32))))
_, pooled_a = encode_images(original, no_positions)
_, pooled_b = encode_images(scrambled, no_positions)
_, pooled_c = encode_images(original, encoder)
_, pooled_d = encode_images(scrambled, encoder)
print("without positions, largest change after shuffling:", np.abs(pooled_a.data - pooled_b.data).max())
print("with positions,    largest change after shuffling:", np.abs(pooled_c.data - pooled_d.data).max().round(4))
assert np.allclose(pooled_a.data, pooled_b.data), "without positions the pooled vector ignores patch order"
assert not np.allclose(pooled_c.data, pooled_d.data), "with positions the order matters"

**Expected behaviour.** Without positions, the largest change is zero or a rounding error around $10^{-16}$; with positions it is clearly larger. Even with positions, this encoder never compares two patches with each other, so it cannot directly see how patches are arranged. Exercise 2 asks what that means for a bar whose thickness matches the patch size.

### 🧪 Experiment 2: patch size, token count and attention cost

Every patch becomes one token for the components that come later. Halving the patch side quadruples the number of tokens, and a dense attention layer compares every token with every other one, so its score matrix grows with the square of the token count. The table uses our 16 × 16 images and then some common real resolutions.

In [ ]:
print(f"{'image':>9s} {'patch':>6s} {'tokens':>7s} {'patch length':>13s} {'attention entries':>18s}")
for side, p in [(16, 2), (16, 4), (16, 8), (16, 16), (224, 16), (224, 14), (336, 14), (448, 16)]:
    tokens = (side // p) ** 2
    print(f"{side:4d}x{side:<4d} {p:6d} {tokens:7d} {p * p * 3:13d} {tokens ** 2:18,d}")

**Expected behaviour.** Our images give 64, 16, 4 or 1 tokens for patches of side 2, 4, 8 or 16. A 224 × 224 image gives 196 tokens with 16-pixel patches and 256 with 14-pixel patches; 336 × 336 with 14-pixel patches gives 576; and 448 × 448 with 16-pixel patches gives 784, four times as many tokens as 224 × 224 and sixteen times as many attention entries. Higher resolution keeps small details but is paid for in tokens and memory.

## Put it together: what does the pooled vector keep?

You now have the full encoder: normalise, cut into patches, embed with positions, one layer per patch, average. Does its pooled vector keep the colour, size and orientation of the bar? To find out, train the encoder together with three small linear heads, one per attribute, using the attribute labels. Each head is a matrix from the 32-number pooled vector to one score per value, trained with cross-entropy:

$$\mathcal{L} = \tfrac{1}{3}\left(\mathcal{L}_{\text{colour}} + \mathcal{L}_{\text{size}} + \mathcal{L}_{\text{orientation}}\right).$$

This is ordinary supervised training with labels, not vision-language learning yet; notebook 2 replaces the labels with captions. The optimiser is **Adam**, given in the next cell: it keeps a running mean of each weight's gradients and of their squares, and scales each step by them, as frameworks such as PyTorch do with `torch.optim.Adam`.

In [ ]:
class Adam:
    """The Adam optimiser for a list of Tensors, as in frameworks such as PyTorch."""

    def __init__(self, params, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        self.params, self.lr, self.betas, self.eps = list(params), lr, betas, eps
        self.m = [np.zeros_like(p.data) for p in self.params]  # running mean of gradients
        self.v = [np.zeros_like(p.data) for p in self.params]  # running mean of squared gradients
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        beta1, beta2 = self.betas
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = beta1 * m + (1 - beta1) * p.grad
            v[...] = beta2 * v + (1 - beta2) * p.grad ** 2
            m_hat, v_hat = m / (1 - beta1 ** self.t), v / (1 - beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [ ]:
probe_encoder = init_image_encoder(seed=0)
head_rng = np.random.default_rng(1)
heads = {attribute: parameter(32, len(values), rng=head_rng, scale=0.1) for attribute, values in ATTRIBUTES.items()}
optimiser = Adam(list(probe_encoder.values()) + list(heads.values()), lr=0.01)
order = np.random.default_rng(2)
losses = []
start = time.perf_counter()
for step in range(150):
    batch = order.choice(len(train_patches), 32, replace=False)
    _, pooled = encode_images(Tensor(train_patches[batch]), probe_encoder)
    loss = (cross_entropy(pooled @ heads["colour"], train_scenes["colour"][batch])
            + cross_entropy(pooled @ heads["size"], train_scenes["size"][batch])
            + cross_entropy(pooled @ heads["orientation"], train_scenes["orientation"][batch])) / 3
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    losses.append(float(loss.data))
print(f"150 steps in {time.perf_counter() - start:.2f} s; loss {np.mean(losses[:10]):.3f} -> {np.mean(losses[-10:]):.4f}")

In [ ]:
def probe_accuracy(patches, scenes):
    _, pooled = encode_images(Tensor(patches), probe_encoder)
    return {attribute: float(((pooled @ heads[attribute]).data.argmax(axis=1) == scenes[attribute]).mean())
            for attribute in ATTRIBUTES}


seen_accuracy = probe_accuracy(prepare(seen_images, mean, std), seen_scenes)
held_out_accuracy = probe_accuracy(prepare(held_out_images, mean, std), held_out_scenes)
print(f"{'accuracy':24s}" + "".join(f"{attribute:>13s}" for attribute in ATTRIBUTES))
for name, accuracy in (("seen combinations", seen_accuracy), ("held-out combinations", held_out_accuracy)):
    print(f"{name:24s}" + "".join(f"{accuracy[attribute]:13.3f}" for attribute in ATTRIBUTES))

plt.figure(figsize=(5, 2.6))
plt.plot(losses)
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("training loss")
plt.title("Encoder and probe heads")
plt.show()

assert np.mean(losses[-10:]) < 0.1 * np.mean(losses[:10]), "the training loss should fall by at least 90%"
assert min(seen_accuracy.values()) >= 0.9, "every attribute should be read correctly on seen combinations"
assert held_out_accuracy["colour"] >= 0.9 and held_out_accuracy["size"] >= 0.85, "colour and size should transfer"
assert held_out_accuracy["orientation"] >= 0.7, "orientation should transfer well above chance (0.5)"
print("All checks passed.")

**Expected behaviour.** Training takes well under a second and the loss falls from about 0.7 to below 0.01. On new images of the seen combinations every attribute is read almost perfectly. On the held-out combinations, colour and size are still close to perfect, but orientation is lower: about 0.92 in our run, and between 0.86 and 1.0 across eight different seeds. The asserted floors (0.9, 0.85 and 0.7) sit below all of those runs.

The gap is the point of the held-out split. Each held-out combination puts familiar values together in a new way, and the encoder's orientation features were learned from bars of other colours and sizes; part of what it learned about orientation is tangled up with those. A random image split would have hidden this completely. Remember too what these numbers do **not** show: they come from 16 × 16 drawings of one bar each, and say nothing about photographs, clutter or text in images.

## Exercises

1. **✍️ `visible_fraction`.** The 🧪 crop cell counted surviving pixels by drawing masks. Compute the same thing from the scene geometry alone. Write `visible_fraction(scenes, crop)`, which returns, for every scene, the fraction of the bar's pixels inside a centred `crop` × `crop` window, as an array of shape (n,). *Hint:* the window covers rows and columns `start` to `start + crop`, with `start = (16 - crop) // 2`; the overlap of two intervals $[a_1, a_2)$ and $[b_1, b_2)$ has length $\max(0, \min(a_2, b_2) - \max(a_1, b_1))$, and the overlap area is the row overlap times the column overlap.
2. **Bars as thick as a patch.** Change `BAR` so that large bars are 4 pixels thick, `{0: (2, 8), 1: (4, 12)}`, rerun the notebook, and compare the held-out orientation accuracy. Before you run it, predict what happens using Experiment 1: when a bar's thickness matches the patch size, many patches are completely filled with colour whatever the orientation.
3. **A different held-out set.** Replace `HELD_OUT_COMBOS` with both yellow vertical bars, `[(3, 0, 1), (3, 1, 1)]`. Which attribute can the training data now predict from the others without looking at the image, and does the held-out accuracy show it?

### ✍️ Exercise 1: `visible_fraction`

Write `visible_fraction(scenes, crop)` as described above. `scenes` is a dictionary like `train_scenes`; use `bar_box` for each bar's height and width. Return a float array of shape (n,) with values between 0 and 1.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def visible_fraction_reference(scenes, crop):
    """The fraction of each bar's pixels that a centred crop x crop window keeps. Returns (n,) floats."""
    start = (IMAGE_SIZE - crop) // 2
    stop = start + crop
    boxes = np.array([bar_box(s, o) for s, o in zip(scenes["size"], scenes["orientation"])])
    top, left = scenes["row"], scenes["col"]
    rows = np.clip(np.minimum(top + boxes[:, 0], stop) - np.maximum(top, start), 0, None)
    cols = np.clip(np.minimum(left + boxes[:, 1], stop) - np.maximum(left, start), 0, None)
    return rows * cols / (boxes[:, 0] * boxes[:, 1])

In [ ]:
def visible_fraction(scenes, crop):
    """The fraction of each bar's pixels that a centred crop x crop window keeps. Returns (n,) floats."""
    # ✍️ Your code here (about seven lines)
    raise NotImplementedError

In [ ]:
visible_fraction = practice("visible_fraction", visible_fraction, visible_fraction_reference, cases=[
    (train_scenes, 12),
    (seen_scenes, 10),
    (held_out_scenes, 16),  # edge case: no crop at all, so every bar is fully visible
])
crop_12 = visible_fraction(train_scenes, 12)
print(f"bars losing pixels to a 12 x 12 crop: {np.mean(crop_12 < 1):.0%}")

**Expected behaviour.** ✅, and the printed share matches the 🧪 crop cell's result for `crop = 12`. Working from the scenes is exact and fast, which is one more reason to keep a manifest next to the pixels.

## Common errors and debugging

- **Normalising with the wrong statistics.** Computing the mean and standard deviation on the test images, or on each image separately, makes training and evaluation inputs inconsistent and leaks test information. Measure them once on the training images and reuse them everywhere.
- **A `patchify` that skips the transpose.** Reshaping (n, H, W, C) straight to (n, N, P·P·C) gives the right shape and the wrong patches: strips of image rows instead of squares. Worse, an `unpatchify` with the matching mistake still passes the round trip. Check the 4 × 4 example by eye: patch 0 must be `[0, 1, 4, 5]`.
- **Mixing layouts.** Our images are channels-last, (n, H, W, C). Code written for channels-first batches, (n, C, H, W), needs a transpose first; otherwise the "channels" axis it normalises is really an image row.
- **`ndarray @ Tensor` raises an error.** NumPy tries to handle the product itself. Wrap the input first, as in `Tensor(patches) @ W`; `Tensor @ ndarray` works because the engine lifts the right-hand side.
- **Gradients that stop.** Writing `Tensor(np.maximum(x.data, 0))` instead of `x.relu()` produces the right numbers but disconnects the graph, so nothing before it learns. `practice_tensor` catches this by comparing gradients as well as values.
- **Reading too much into the probe.** High probe accuracy shows that the pooled vector contains the attributes on this generated data. It does not show that the encoder would recognise real objects, or that a language model will use those features.

## Summary

- Paired data is only useful when the caption describes what the pixels show. Generated scenes make that checkable, and a manifest of scenes travels with the images.
- A held-out *combination* split tests whether familiar attributes are recognised in new pairings, which a random image split cannot test.
- Normalisation uses training statistics only, and patches turn an image of shape (n, H, W, C) into a sequence of shape (n, N, P·P·C), with $N = (H/P)(W/P)$.
- A patch embedding is one shared linear map plus a learned position vector per patch. Without positions, a pooled encoder ignores the arrangement of its patches.
- The pooled vector of a tiny encoder already separates colour, size and orientation, but held-out combinations expose weaker orientation transfer.

## Next

In **notebook 2** you replace the attribute labels with the captions themselves: a bag-of-tokens text encoder and the contrastive loss of CLIP pull each image's vector towards its caption's vector, and you measure retrieval on the held-out combinations.